In [0]:
%python
from pyspark.sql import functions as F

bronze = spark.table("workspace.energy.bronze_generation")

meta_cols = ["datum_von", "datum_bis", "ingested_at", "source_file"]
source_cols = [c for c in bronze.columns if c not in meta_cols]

print(source_cols)

['biomasse', 'wasserkraft', 'wind_offshore', 'wind_onshore', 'photovoltaik', 'sonstige_erneuerbare', 'kernenergie', 'braunkohle', 'steinkohle', 'erdgas', 'pumpspeicher', 'sonstige_konventionelle']


In [0]:
%python
df = (bronze
      .withColumn("ts_local", F.to_timestamp("datum_von", "dd.MM.yyyy HH:mm"))
      .withColumn("ts_utc", F.to_utc_timestamp("ts_local", "Europe/Berlin")))

display(df.select("datum_von", "ts_local", "ts_utc").limit(5))

datum_von,ts_local,ts_utc
01.10.2020 00:00,2020-10-01T00:00:00.000Z,2020-09-30T22:00:00.000Z
01.10.2020 01:00,2020-10-01T01:00:00.000Z,2020-09-30T23:00:00.000Z
01.10.2020 02:00,2020-10-01T02:00:00.000Z,2020-10-01T00:00:00.000Z
01.10.2020 03:00,2020-10-01T03:00:00.000Z,2020-10-01T01:00:00.000Z
01.10.2020 04:00,2020-10-01T04:00:00.000Z,2020-10-01T02:00:00.000Z


In [0]:
%python
for c in source_cols:
    s = F.regexp_replace(F.regexp_replace(F.col(c), r"\.", ""), ",", ".")
    df = df.withColumn(c, F.when(s.rlike(r"^-?\d+(\.\d+)?$"), s.cast("double")))

display(df.select("ts_utc", "biomasse", "wind_onshore", "photovoltaik").limit(5))
df.printSchema()

ts_utc,biomasse,wind_onshore,photovoltaik
2020-09-30T22:00:00.000Z,4160.5,7087.75,0.0
2020-09-30T23:00:00.000Z,4147.5,7454.5,0.0
2020-10-01T00:00:00.000Z,4136.0,7704.75,0.0
2020-10-01T01:00:00.000Z,4130.5,8053.5,0.0
2020-10-01T02:00:00.000Z,4149.5,8306.5,0.0


root
 |-- datum_von: string (nullable = true)
 |-- datum_bis: string (nullable = true)
 |-- biomasse: double (nullable = true)
 |-- wasserkraft: double (nullable = true)
 |-- wind_offshore: double (nullable = true)
 |-- wind_onshore: double (nullable = true)
 |-- photovoltaik: double (nullable = true)
 |-- sonstige_erneuerbare: double (nullable = true)
 |-- kernenergie: double (nullable = true)
 |-- braunkohle: double (nullable = true)
 |-- steinkohle: double (nullable = true)
 |-- erdgas: double (nullable = true)
 |-- pumpspeicher: double (nullable = true)
 |-- sonstige_konventionelle: double (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ts_local: timestamp (nullable = true)
 |-- ts_utc: timestamp (nullable = true)



In [0]:
%python
stack_expr = "stack({}, {}) as (energy_source, mwh)".format(
    len(source_cols),
    ", ".join(f"'{c}', {c}" for c in source_cols)
)

long_df = df.select("ts_utc", "ts_local", F.expr(stack_expr))

print(long_df.count(), "rows")
display(long_df.orderBy("ts_utc").limit(12))

637344 rows


ts_utc,ts_local,energy_source,mwh
2020-09-30T22:00:00.000Z,2020-10-01T00:00:00.000Z,wasserkraft,1930.75
2020-09-30T22:00:00.000Z,2020-10-01T00:00:00.000Z,steinkohle,3330.25
2020-09-30T22:00:00.000Z,2020-10-01T00:00:00.000Z,biomasse,4160.5
2020-09-30T22:00:00.000Z,2020-10-01T00:00:00.000Z,wind_onshore,7087.75
2020-09-30T22:00:00.000Z,2020-10-01T00:00:00.000Z,wind_offshore,5396.75
2020-09-30T22:00:00.000Z,2020-10-01T00:00:00.000Z,pumpspeicher,1404.75
2020-09-30T22:00:00.000Z,2020-10-01T00:00:00.000Z,braunkohle,12365.5
2020-09-30T22:00:00.000Z,2020-10-01T00:00:00.000Z,kernenergie,6626.5
2020-09-30T22:00:00.000Z,2020-10-01T00:00:00.000Z,sonstige_erneuerbare,177.0
2020-09-30T22:00:00.000Z,2020-10-01T00:00:00.000Z,photovoltaik,0.0


In [0]:
%python
renewables = ["biomasse", "wasserkraft", "wind_offshore", "wind_onshore",
              "photovoltaik", "sonstige_erneuerbare"]

long_df = long_df.withColumn(
    "category",
    F.when(F.col("energy_source").isin(renewables), "renewable")
     .when(F.col("energy_source") == "pumpspeicher", "storage")
     .otherwise("conventional")
)

display(long_df.groupBy("category", "energy_source").count().orderBy("category"))

category,energy_source,count
conventional,steinkohle,53112
conventional,erdgas,53112
conventional,braunkohle,53112
conventional,kernenergie,53112
conventional,sonstige_konventionelle,53112
renewable,wind_offshore,53112
renewable,sonstige_erneuerbare,53112
renewable,biomasse,53112
renewable,photovoltaik,53112
renewable,wasserkraft,53112


In [0]:
%python
silver = (long_df
          .filter("ts_utc IS NOT NULL")
          .dropDuplicates(["ts_utc", "energy_source"])
          .withColumn("silver_loaded_at", F.current_timestamp()))

total = silver.count()
null_vals = silver.filter("mwh IS NULL").count()
neg_vals = silver.filter("mwh < 0").count()
n_sources = silver.select("energy_source").distinct().count()

print(f"rows: {total}, null mwh: {null_vals}, negative mwh: {neg_vals}, sources: {n_sources}")

assert total > 0, "Silver is empty"
assert n_sources == len(source_cols), "Missing energy sources"
assert null_vals / total < 0.05, "Too many null values"
print("quality checks passed")

rows: 637272, null mwh: 23913, negative mwh: 0, sources: 12
quality checks passed


In [0]:
%python
from delta.tables import DeltaTable

SILVER = "workspace.energy.silver_generation"

if not spark.catalog.tableExists(SILVER):
    silver.write.saveAsTable(SILVER)
else:
    (DeltaTable.forName(spark, SILVER).alias("t")
        .merge(silver.alias("s"),
               "t.ts_utc = s.ts_utc AND t.energy_source = s.energy_source")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

result = spark.table(SILVER)
print("silver rows:", result.count())
display(result.orderBy("ts_utc").limit(10))

silver rows: 637272


ts_utc,ts_local,energy_source,mwh,category,silver_loaded_at
2020-09-30T22:00:00.000Z,2020-10-01T00:00:00.000Z,steinkohle,3330.25,conventional,2026-10-27T10:09:15.733Z
2020-09-30T22:00:00.000Z,2020-10-01T00:00:00.000Z,pumpspeicher,1404.75,storage,2026-10-27T10:09:15.733Z
2020-09-30T22:00:00.000Z,2020-10-01T00:00:00.000Z,photovoltaik,0.0,renewable,2026-10-27T10:09:15.733Z
2020-09-30T22:00:00.000Z,2020-10-01T00:00:00.000Z,sonstige_erneuerbare,177.0,renewable,2026-10-27T10:09:15.733Z
2020-09-30T22:00:00.000Z,2020-10-01T00:00:00.000Z,wind_offshore,5396.75,renewable,2026-10-27T10:09:15.733Z
2020-09-30T22:00:00.000Z,2020-10-01T00:00:00.000Z,wind_onshore,7087.75,renewable,2026-10-27T10:09:15.733Z
2020-09-30T22:00:00.000Z,2020-10-01T00:00:00.000Z,erdgas,7329.5,conventional,2026-10-27T10:09:15.733Z
2020-09-30T22:00:00.000Z,2020-10-01T00:00:00.000Z,wasserkraft,1930.75,renewable,2026-10-27T10:09:15.733Z
2020-09-30T22:00:00.000Z,2020-10-01T00:00:00.000Z,biomasse,4160.5,renewable,2026-10-27T10:09:15.733Z
2020-09-30T22:00:00.000Z,2020-10-01T00:00:00.000Z,kernenergie,6626.5,conventional,2026-10-27T10:09:15.733Z


In [0]:
%python
display(spark.sql("""
    SELECT energy_source, COUNT(*) AS null_rows
    FROM workspace.energy.silver_generation
    WHERE mwh IS NULL
    GROUP BY energy_source
    ORDER BY null_rows DESC
"""))

energy_source,null_rows
kernenergie,23913


In [0]:
%python
display(spark.sql("""
    SELECT MIN(ts_utc) AS first_null, MAX(ts_utc) AS last_null
    FROM workspace.energy.silver_generation
    WHERE mwh IS NULL
"""))

first_null,last_null
2024-01-30T11:00:00.000Z,2026-10-22T21:00:00.000Z


All 23,913 null rows belong to nuclear (about 996 days of hourly data). That fits Germany’s nuclear phase-out in 2023: after the last reactors shut down, SMARD has no values for nuclear. It’s a good finding for your README: “Nulls occur only in kernenergie after the phase-out, so they were treated as no generation.”